# Music Annotation Project V8.8B
## Notebook 53R3: Notebook 51R2 Exact Regression Compatibility Builder

**Runtime:** CPU. Run all in a fresh Colab runtime.

### Purpose
This notebook exists to close the Stage F regression question by reproducing the **actual executable behaviour** of Notebook 51R2 on the frozen regression roster.

### Critical lineage note
Notebook 51R2 contains an internal inconsistency: its pre-registration and final narrative describe a REMOVE-only policy, but its executable mask-construction cell decodes both Notebook 50 decision classes and constructs `M5 = M3 & ~SAFE_REMOVE & ~SAFE_PROTECT`.

This notebook therefore has two explicitly separated meanings:

1. **REGISTERED_REPLAY**: reproduces the historical Notebook 51R2 executable candidate exactly, including exclusion of decision code 2 from the Telea residual target. This route is for Stage F equivalence only.
2. **OPERATIONAL_REMOVE_ONLY**: retains the V8.8 policy contract `M5_REMOVE = M3 & ~SAFE_REMOVE`; SAFE_PROTECT is not operationalised.

The default is `REGISTERED_REPLAY` because the immediate objective is to determine whether Stage F can reach exact equality with the durable Notebook 51R2 golden masters.

No clean truth is used during reconstruction. Outside-authority bytes remain immutable.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow scikit-image


## 1. Configuration and run identity


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, sys, platform
import cv2
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
N45_ROOT = ROOT / 'observed_print_classifier_v8_6_notebook45'
N46_ROOT = ROOT / 'observed_print_classifier_v8_6_notebook46'
N50_ROOT = ROOT / 'observable_residual_mask_m5_v8_7'
N51_ROOT = ROOT / 'observed_print_classifier_v8_7_notebook51'
OUT_ROOT = ROOT / 'observed_print_classifier_v8_8_notebook53R3'

# Set explicit overrides only when needed.
N45_RUN_OVERRIDE = None
N46_RUN_OVERRIDE = None
N50_RUN_OVERRIDE = None
N51_RUN_OVERRIDE = None

# Stage-F compatibility route. Do not change while validating Notebook 51R2 equivalence.
INPUT_MODE = 'REGISTERED_REPLAY'
VALID_MODES = {'REGISTERED_REPLAY', 'OPERATIONAL_REMOVE_ONLY'}
if INPUT_MODE not in VALID_MODES:
    raise RuntimeError(f'INPUT_MODE must be one of {sorted(VALID_MODES)}')

INPAINT_RADIUS = 3
SAFE_REMOVE_CODE = 1
SAFE_PROTECT_CODE = 2

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_v8_8b_notebook53R3'
OUT = OUT_ROOT / RUN_ID
PAGES_OUT = OUT / 'pages'
TABLES = OUT / 'tables'
AUDIT = OUT / 'audit'
UPLOAD = OUT / 'upload'
for p in [OUT, PAGES_OUT, TABLES, AUDIT, UPLOAD]:
    p.mkdir(parents=True, exist_ok=False)


def require(condition, message):
    if not condition:
        try:
            (AUDIT / 'fatal_failure.json').write_text(json.dumps({
                'utc': datetime.now(timezone.utc).isoformat(),
                'message': str(message),
            }, indent=2))
        except Exception:
            pass
        raise RuntimeError(message)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()


def array_hash(a):
    a = np.ascontiguousarray(a)
    h = hashlib.sha256()
    h.update(str(a.dtype).encode())
    h.update(str(a.shape).encode())
    h.update(a.tobytes())
    return h.hexdigest()


def newest_complete(root, required):
    hits = []
    if Path(root).exists():
        for p in Path(root).iterdir():
            if p.is_dir() and all((p / r).exists() for r in required):
                hits.append(p)
    return max(hits, key=lambda p: p.stat().st_mtime) if hits else None

print('Mode:', INPUT_MODE)
print('Output:', OUT)


## 2. Resolve frozen Notebook 45, 46R6, 50 and 51R2 runs


In [ ]:
REQ45 = ['mask_lock.json', 'source_record.json', 'tables/region_manifest.parquet']
REQ46 = ['source_record.json', 'audit/output_completion.json']
REQ50 = ['notebook50_verdict.json', 'tables/m5_map_inventory.csv']
REQ51 = ['04_candidate_lock/candidate_inventory.csv', '04_candidate_lock/candidate_lock.json']

N45 = Path(N45_RUN_OVERRIDE) if N45_RUN_OVERRIDE else newest_complete(N45_ROOT, REQ45)
N46 = Path(N46_RUN_OVERRIDE) if N46_RUN_OVERRIDE else newest_complete(N46_ROOT, REQ46)
N50 = Path(N50_RUN_OVERRIDE) if N50_RUN_OVERRIDE else newest_complete(N50_ROOT, REQ50)
N51 = Path(N51_RUN_OVERRIDE) if N51_RUN_OVERRIDE else newest_complete(N51_ROOT, REQ51)

require(N45 is not None, 'No complete Notebook45 run found.')
require(N46 is not None, 'No complete Notebook46R6 run found.')
require(N50 is not None, 'No complete Notebook50 run found.')
require(N51 is not None, 'No complete Notebook51R2 run found.')

print('Notebook45:', N45)
print('Notebook46:', N46)
print('Notebook50:', N50)
print('Notebook51R2:', N51)

regions = pd.read_parquet(N45 / 'tables/region_manifest.parquet')
need = {'page','source_group','region_id','y0','y1','x0','x1'}
require(need.issubset(regions.columns), f'Notebook45 region manifest missing {sorted(need-set(regions.columns))}')
regions['page'] = regions.page.astype(str)
regions['source_group'] = regions.source_group.astype(str)
require(not regions[['page','region_id']].duplicated().any(), 'Duplicate Notebook45 region identity.')

map_inventory = pd.read_csv(N50 / 'tables/m5_map_inventory.csv', dtype=str, keep_default_na=False)
require({'page','file'}.issubset(map_inventory.columns), 'Notebook50 M5 inventory lacks page/file.')
map_inventory['page'] = map_inventory.page.astype(str)

n51_candidates = pd.read_csv(N51 / '04_candidate_lock' / 'candidate_inventory.csv', dtype=str, keep_default_na=False)
for c in ['page','arm','restorer','candidate']:
    require(c in n51_candidates.columns, f'Notebook51 candidate inventory lacks {c}')
    n51_candidates[c] = n51_candidates[c].astype(str).str.strip()

pages = sorted(regions.page.unique())
print('Regression pages:', len(pages))


## 3. Load authoritative page inputs and Notebook 45 regional masks

The classical Notebook 51R2 Telea path uses the full-page regional `M3.png` masks directly. It does **not** turn region bounding boxes into rectangular masks and does not clip these masks to `y0:y1, x0:x1`.


In [ ]:
source45 = json.loads((N45 / 'source_record.json').read_text())
source_v82 = Path(source45['source_v82'])

# Frozen Notebook46 base/protected pages.
def resolve_base(page):
    candidates = [
        N46 / 'whole_page_candidates' / page / 'v7_8.png',
        source_v82 / 'whole_pages' / page / 'v7_8.png',
    ]
    for p in candidates:
        if p.exists():
            return p
    raise RuntimeError(f'{page}: cannot resolve frozen V7.8 base.')


def resolve_protected(page):
    p = N46 / 'whole_page_candidates' / page / 'protected_print.png'
    require(p.exists(), f'{page}: missing protected_print.png: {p}')
    return p


def union_arm(page, arm='M3'):
    arr = None
    for _, r in regions[regions.page == page].iterrows():
        p = N45 / 'masks' / page / str(r.region_id).replace(':','_') / f'{arm}.png'
        m = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        require(m is not None, f'Missing/unreadable Notebook45 mask: {p}')
        b = m > 0
        arr = b.copy() if arr is None else (arr | b)
    require(arr is not None, f'{page}: no regional {arm} masks.')
    return arr

page_obs = {}
page_base = {}
page_protected = {}
page_m3 = {}

for page in pages:
    obs_path = source_v82 / 'whole_pages' / page / 'observed.png'
    base_path = resolve_base(page)
    protected_path = resolve_protected(page)
    obs = cv2.imread(str(obs_path), cv2.IMREAD_GRAYSCALE)
    base = cv2.imread(str(base_path), cv2.IMREAD_GRAYSCALE)
    protected = cv2.imread(str(protected_path), cv2.IMREAD_GRAYSCALE)
    m3 = union_arm(page, 'M3')
    require(obs is not None and base is not None and protected is not None, f'{page}: missing page inputs.')
    require(obs.shape == base.shape == protected.shape == m3.shape, f'{page}: page input shape mismatch.')
    page_obs[page] = obs
    page_base[page] = base
    page_protected[page] = protected > 0
    page_m3[page] = m3

print('Loaded authoritative page state:', len(page_obs))


## 4. Decode Notebook 50 decision maps exactly

Notebook 50 may have saved a decision canvas smaller than the physical page. The historical Notebook 51R2 implementation restores that canvas origin-aligned into the authoritative full-page shape. Decision code 1 is SAFE_REMOVE; decision code 2 is SAFE_PROTECT.


In [ ]:
def resolve_notebook50_decisions(page, page_shape):
    q = map_inventory[map_inventory.page == page]
    require(len(q) == 1, f'{page}: expected exactly one Notebook50 M5 inventory row, found {len(q)}.')
    p = Path(q.iloc[0]['file'])
    if not p.exists():
        # Narrow deterministic fallback under N50 only.
        hits = [x for x in N50.rglob(p.name) if x.is_file()]
        require(len(hits) == 1, f'{page}: could not uniquely resolve Notebook50 decision file {p.name}: {hits}')
        p = hits[0]
    require(p.suffix.lower() == '.npz', f'{page}: expected Notebook50 decision NPZ, got {p}')
    with np.load(p, allow_pickle=False) as z:
        require('m5_decision_map' in z.files, f'{page}: {p.name} lacks m5_decision_map')
        small = np.asarray(z['m5_decision_map'])
    require(small.ndim == 2, f'{page}: decision map is not 2-D: {small.shape}')
    require(small.shape[0] <= page_shape[0] and small.shape[1] <= page_shape[1],
            f'{page}: decision map {small.shape} exceeds page {page_shape}')
    decision = np.zeros(page_shape, dtype=small.dtype)
    decision[:small.shape[0], :small.shape[1]] = small
    sr = decision == SAFE_REMOVE_CODE
    sp = decision == SAFE_PROTECT_CODE
    return sr, sp, p, small.shape

mask_rows = []
page_safe_remove = {}
page_safe_protect = {}
page_m5 = {}

for page in pages:
    m3 = page_m3[page]
    sr, sp, decision_path, small_shape = resolve_notebook50_decisions(page, m3.shape)
    require(not np.any(sr & ~m3), f'{page}: SAFE_REMOVE outside M3')
    require(not np.any(sp & ~m3), f'{page}: SAFE_PROTECT outside M3')
    require(not np.any(sr & sp), f'{page}: SAFE_REMOVE/SAFE_PROTECT overlap')

    if INPUT_MODE == 'REGISTERED_REPLAY':
        # Historical executable Notebook51R2 behaviour.
        m5 = m3 & ~sr & ~sp
    else:
        # Commissioned V8.8 REMOVE-only policy.
        m5 = m3 & ~sr

    page_safe_remove[page] = sr
    page_safe_protect[page] = sp
    page_m5[page] = m5
    mask_rows.append({
        'page': page,
        'M3_pixels': int(m3.sum()),
        'SAFE_REMOVE_pixels': int(sr.sum()),
        'SAFE_PROTECT_decision_pixels': int(sp.sum()),
        'M5_pixels': int(m5.sum()),
        'mode': INPUT_MODE,
        'decision_canvas_shape': str(tuple(small_shape)),
        'page_shape': str(tuple(m3.shape)),
        'decision_path': str(decision_path),
        'decision_sha256': sha256_file(decision_path),
    })

mask_df = pd.DataFrame(mask_rows)
mask_df.to_csv(TABLES / 'mask_inventory.csv', index=False)
display(mask_df)
print(mask_df[['M3_pixels','SAFE_REMOVE_pixels','SAFE_PROTECT_decision_pixels','M5_pixels']].sum())


## 5. Exact Notebook 51R2 Telea reconstruction and candidate composition

The historical candidate base is the frozen V7.8 page with frozen protected-print pixels restored from the observed page. Only regional M5 pixels are replaced by the result of full-page `cv2.inpaint(observed, regional_mask, 3, TELEA)`.

**Important:** excluding a decision-code-2 coordinate from M5 does not itself copy observed bytes there. The coordinate simply never enters the new M5 Telea replacement and therefore retains the historical candidate-base treatment.


In [ ]:
def reconstruct_telea_exact(obs, mask):
    m = (mask > 0).astype(np.uint8) * 255
    if not m.any():
        return obs.copy()
    cand = cv2.inpaint(obs, m, INPAINT_RADIUS, cv2.INPAINT_TELEA)
    out = obs.copy()
    out[m > 0] = cand[m > 0]
    require(np.array_equal(out[m == 0], obs[m == 0]), 'Telea outside-mask drift.')
    return out


def resolve_n51_reference(page):
    # Historical files used both labels in forensic work. Prefer inventory semantics and fail on ambiguity.
    q = n51_candidates[
        (n51_candidates.page == page)
        & (n51_candidates.restorer == 'R2_telea')
        & (n51_candidates.arm.isin(['M5','M5_REMOVE']))
    ].copy()
    require(len(q) == 1, f'{page}: expected one Notebook51 R2 Telea M5/M5_REMOVE candidate, found {len(q)}')
    p = Path(q.iloc[0].candidate)
    if not p.exists():
        fallback = N51 / '03_candidates' / page / p.name
        if fallback.exists():
            p = fallback
        else:
            hits = [x for x in N51.rglob(p.name) if x.is_file() and x.parent.name == page]
            require(len(hits) == 1, f'{page}: cannot uniquely resolve Notebook51 reference: {hits}')
            p = hits[0]
    return p, str(q.iloc[0].arm)

page_rows = []
region_rows = []

for page in pages:
    obs = page_obs[page]
    base = page_base[page]
    protected = page_protected[page]
    m3 = page_m3[page]
    sr = page_safe_remove[page]
    sp = page_safe_protect[page]
    m5 = page_m5[page]

    candidate = base.copy()
    candidate[protected] = obs[protected]
    regional_union = np.zeros_like(m5, dtype=bool)

    pg = regions[regions.page == page]
    for _, r in pg.iterrows():
        region_id = str(r.region_id)
        mp = N45 / 'masks' / page / region_id.replace(':','_') / 'M3.png'
        rm = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
        require(rm is not None and rm.shape == obs.shape, f'{page}/{region_id}: invalid regional M3 {mp}')
        region_m3 = rm > 0
        regional = m5 & region_m3
        regional_union |= regional
        if regional.any():
            recon = reconstruct_telea_exact(obs, regional)
            candidate[regional] = recon[regional]
        region_rows.append({
            'page': page,
            'region_id': region_id,
            'regional_M3_pixels': int(region_m3.sum()),
            'regional_M5_pixels': int(regional.sum()),
            'regional_mask_sha256': array_hash(regional.astype(np.uint8)),
            'm3_path': str(mp),
        })

    require(np.array_equal(regional_union, m5), f'{page}: regional M5 union differs from page M5')

    ref_path, ref_arm = resolve_n51_reference(page)
    ref = cv2.imread(str(ref_path), cv2.IMREAD_GRAYSCALE)
    require(ref is not None and ref.shape == candidate.shape, f'{page}: invalid Notebook51 reference {ref_path}')

    diff = candidate != ref
    diff_count = int(diff.sum())
    max_abs = int(np.max(np.abs(candidate.astype(np.int16) - ref.astype(np.int16))))

    d = PAGES_OUT / page
    d.mkdir(parents=True, exist_ok=False)
    cv2.imwrite(str(d / 'observed.png'), obs)
    cv2.imwrite(str(d / 'm3.png'), m3.astype(np.uint8) * 255)
    cv2.imwrite(str(d / 'safe_remove.png'), sr.astype(np.uint8) * 255)
    cv2.imwrite(str(d / 'safe_protect_decision.png'), sp.astype(np.uint8) * 255)
    cv2.imwrite(str(d / 'm5.png'), m5.astype(np.uint8) * 255)
    cv2.imwrite(str(d / 'corrected.png'), candidate)
    cv2.imwrite(str(d / 'reference_51R2.png'), ref)
    cv2.imwrite(str(d / 'difference_mask.png'), diff.astype(np.uint8) * 255)

    page_rows.append({
        'page': page,
        'mode': INPUT_MODE,
        'M3_pixels': int(m3.sum()),
        'SAFE_REMOVE_pixels': int(sr.sum()),
        'SAFE_PROTECT_decision_pixels': int(sp.sum()),
        'M5_pixels': int(m5.sum()),
        'regional_union_pixels': int(regional_union.sum()),
        'N51_arm_label': ref_arm,
        'N51_exact': diff_count == 0,
        'different_pixels': diff_count,
        'max_absolute_difference': max_abs,
        'built_array_sha256': array_hash(candidate),
        'reference_array_sha256': array_hash(ref),
        'reference_path': str(ref_path),
    })
    print('Built', page, 'M3', int(m3.sum()), 'SAFE_REMOVE', int(sr.sum()),
          'SAFE_PROTECT decision', int(sp.sum()), 'M5', int(m5.sum()),
          'N51 exact', diff_count == 0, 'diff', diff_count)

page_df = pd.DataFrame(page_rows)
region_df = pd.DataFrame(region_rows)
page_df.to_csv(TABLES / 'page_equivalence.csv', index=False)
region_df.to_csv(TABLES / 'regional_masks.csv', index=False)
display(page_df)


## 6. Fail-closed Stage F verdict

For `REGISTERED_REPLAY`, PASS means this implementation reproduced every durable Notebook 51R2 R2-Telea corrected page exactly. This is an implementation-equivalence result only. It must not be misreported as validation of SAFE_PROTECT for V8.8 operational use.


In [ ]:
failed = page_df[~page_df.N51_exact].copy()
all_exact = bool(len(page_df) and page_df.N51_exact.all())

verdict = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'version': '53R3-v8.8b-notebook51R2-regression-compatibility',
    'mode': INPUT_MODE,
    'pages': int(len(page_df)),
    'stage_F_corrected_page_exact': all_exact,
    'failed_pages': failed.page.astype(str).tolist(),
    'total_different_pixels': int(page_df.different_pixels.sum()),
    'maximum_absolute_difference': int(page_df.max_absolute_difference.max()) if len(page_df) else None,
    'historical_51R2_execution_quirk': (
        'Notebook51R2 pre-registration/final narrative state REMOVE-only, '
        'but its executable mask-generation cell excludes decision codes 1 and 2 from M5.'
    ),
    'safe_protect_operationally_authorised': False,
    'registered_replay_uses_decision_code_2_exclusion': INPUT_MODE == 'REGISTERED_REPLAY',
    'verdict': 'PASS_STAGE_F_REGRESSION_COMPATIBILITY' if all_exact else 'FAIL_STAGE_F_REGRESSION_COMPATIBILITY',
}
(OUT / 'stage_f_verdict.json').write_text(json.dumps(verdict, indent=2))
print(json.dumps(verdict, indent=2))

if not all_exact:
    failed.to_csv(TABLES / 'failed_pages.csv', index=False)
    raise RuntimeError(
        f'Stage F remains failed: {len(failed)} page(s), '
        f'{int(page_df.different_pixels.sum())} differing pixels. '
        f'See {TABLES / "failed_pages.csv"}'
    )

print('STAGE F REGRESSION COMPATIBILITY PASSED: zero different corrected-page pixels.')


## 7. Identity and curated upload package


In [ ]:
environment = {
    'python': sys.version,
    'opencv': cv2.__version__,
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'platform': platform.platform(),
}
(OUT / 'software_environment.json').write_text(json.dumps(environment, indent=2))

identity = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'version': '53R3-v8.8b-notebook51R2-regression-compatibility',
    'mode': INPUT_MODE,
    'notebook45': str(N45),
    'notebook46': str(N46),
    'notebook50': str(N50),
    'notebook51R2': str(N51),
    'pages': len(pages),
    'inpaint_radius': INPAINT_RADIUS,
    'safe_remove_code': SAFE_REMOVE_CODE,
    'safe_protect_code': SAFE_PROTECT_CODE,
    'operational_safe_protect_authorised': False,
    'software': environment,
}
(OUT / 'experiment_identity.json').write_text(json.dumps(identity, indent=2))

summary_lines = [
    'NOTEBOOK 53R3 - NOTEBOOK 51R2 EXACT REGRESSION COMPATIBILITY',
    '',
    f'Run: {OUT}',
    f'Mode: {INPUT_MODE}',
    f'Pages: {len(page_df)}',
    f'Exact pages: {int(page_df.N51_exact.sum())}',
    f'Different pixels: {int(page_df.different_pixels.sum())}',
    '',
    'Historical lineage note:',
    'Notebook 51R2 declares REMOVE-only in its pre-registration/final narrative,',
    'but its executable M5 cell removes both decision code 1 and code 2 from the residual Telea target.',
    'REGISTERED_REPLAY reproduces that executable behaviour only to close the exact regression question.',
    'SAFE_PROTECT remains unauthorised for the V8.8 operational policy.',
]
(OUT / 'executive_summary.txt').write_text(chr(10).join(summary_lines))

required = [
    OUT / 'executive_summary.txt',
    OUT / 'stage_f_verdict.json',
    OUT / 'experiment_identity.json',
    OUT / 'software_environment.json',
    TABLES / 'mask_inventory.csv',
    TABLES / 'page_equivalence.csv',
    TABLES / 'regional_masks.csv',
]
for p in required:
    require(p.exists(), f'Missing upload artefact: {p}')
    shutil.copy2(p, UPLOAD / p.name)

upload_rows = []
for p in sorted(UPLOAD.iterdir()):
    if p.is_file():
        upload_rows.append({'file': p.name, 'bytes': p.stat().st_size, 'sha256': sha256_file(p)})
pd.DataFrame(upload_rows).to_csv(UPLOAD / 'upload_manifest.csv', index=False)
print(chr(10).join(summary_lines))
print('Upload package directory:', UPLOAD)


## Interpretation after execution

- If `REGISTERED_REPLAY` produces zero corrected-page differences, the Stage F *historical implementation-equivalence* problem is closed.
- That result also documents a lineage inconsistency in Notebook 51R2: the executable candidate used decision code 2 as a residual-mask exclusion even though the retained V8.7/V8.8 policy is REMOVE-only.
- Do **not** silently carry that historical code-2 exclusion into the V8.8 operational policy. The operational route remains `M5_REMOVE = M3 & ~SAFE_REMOVE` unless the project explicitly commissions a policy change.
- Notebook 54 should distinguish historical 51R2 corrected-page replay equivalence from V8.8 policy fidelity instead of requiring a REMOVE-only implementation to reproduce a golden master generated by a combined exclusion implementation.
